# RaceBench — comparison report

One row per (task, strategy): correctness, cost, and the false-positive stall rate nobody else reports.
Point `RUN_DIRS` at one or more `results/<run_id>` directories and run all cells.

In [ ]:
from pathlib import Path

import pandas as pd

from analysis.metrics import aggregate, run_dataframe
from analysis.plots import make_all_plots

RUN_DIRS = ["../results/smoke-scripted", "../results/smoke-clobber"]

df = pd.concat([run_dataframe(Path(d)) for d in RUN_DIRS], ignore_index=True)
print(f"{len(df)} trials")
df.head()

## The comparison table

The deliverable: every coordination strategy on the same tasks, same metrics.

In [ ]:
agg = aggregate(df)
agg

## Headline: false-positive stalls

`t2_benign_overlap` is deliberately benign — two agents edit disjoint functions in the same file. Any coordination event there is pure overhead. File-level locking stalls; symbol-level claims do not.

In [ ]:
benign = df[df["benign"]]
benign.groupby("strategy")[["stall_events", "fp_stall_events", "correct"]].mean()

## Safety: does the strategy prevent silent lost updates?

On `t1_stale_read` with stale whole-file writes, `naive` silently drops one agent's work; `git_hash` either merges it or surfaces a conflict — never a silent loss.

In [ ]:
t1 = df[df["task"] == "t1_stale_read"]
t1.groupby(["strategy", "model"])[["correct", "oracle_passed", "oracle_total"]].mean()

In [ ]:
figures = make_all_plots(agg)
from IPython.display import Image, display
for f in figures:
    display(Image(str(f)))